# Staged LightGCN Scale Experiment

This notebook compares completed experiment stages. The stages diagnose candidate coverage, metric stability, and training duration separately before a final configuration is selected.

In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
comparison_path = (
    project_root / "outputs/lightgcn_scale_experiment/scale_comparison.csv.gz"
)
if not comparison_path.exists():
    raise FileNotFoundError(
        "Run scripts/run_lightgcn_scale_experiment.py before this notebook."
    )
results = pd.read_csv(comparison_path)

## 1. Compare stage scope

One row per stage is enough to inspect graph size and validation-target coverage because those values are shared by all ranking methods in that stage.

In [ ]:
scope_columns = [
    "stage",
    "selected_pair_count",
    "evaluated_pair_count",
    "graph_user_count",
    "graph_movie_count",
    "positive_edge_count",
    "training_steps",
    "validation_target_coverage",
]
results[scope_columns].drop_duplicates().reset_index(drop=True)

## 2. Track the primary metric

The goal is not merely to make minimum-member NDCG non-zero. A useful stage should improve it consistently across methods and retain enough evaluable pairs to support a comparison.

In [ ]:
results.pivot(
    index="stage",
    columns="method",
    values="mean_minimum_ndcg_at_10",
)

## 3. Inspect the trade-off

Conflict-aware ranking should be judged jointly: minimum-member NDCG should rise without an unacceptable reduction in average NDCG or catalogue coverage. The NDCG gap is descriptive and lower is better.

In [ ]:
metric_columns = [
    "stage",
    "method",
    "mean_minimum_ndcg_at_10",
    "mean_average_ndcg_at_10",
    "mean_ndcg_gap_at_10",
    "catalogue_coverage_at_10",
]
results[metric_columns].sort_values(
    ["stage", "mean_minimum_ndcg_at_10", "mean_average_ndcg_at_10"],
    ascending=[True, False, False],
)

## 4. Preserve the evaluation boundary

Choose graph scale, training duration, and conflict weight using validation results only. After fixing them, retrain under the declared protocol and evaluate the test split once. Repeated test inspection would turn the test set into another validation set.